# 第07章 信任的边界

> 使用方法：保留本模板，先复制到本章 `工作/` 目录，再将文件名前缀 `模板-A组-` / `模板-B组-` / `模板-C组-` 改为本章对应的 `A1组-` / `B2组-` / `C1组-` 等。按章节分工表确认编号，正文内的同组文件链接也改到副本；公开只填组别。复用导入图时，将本章 `模板/图片/` 中的图复制到 `工作/图片/`，图片链接使用 `图片/文件名`。

![三格漫画：训练区间像已绘好的地图，新区间像未知道路，机器人询问信任的适用范围。](图片/A-章节导入.svg)

*图07-1 · 地图之外，还能相信吗？初始化概念示意；适用条件见下文，不作为实测结果。*

## 先看一幕：地图之外，还能相信吗？

机器人在熟悉的街区导航得很准。一天，它走出地图覆盖范围，仍然自信地延长路线：熟悉区域里的好表现，究竟能支撑多远的信任？

**先猜，再验证：** 用 $[-1,1]$ 内的数据拟合 $\sin(x)$ 后，为什么仍需单独评估 $[2,3]$ 上的误差？训练误差小，缺少了哪些条件才谈得上域外保证？

**把故事翻译成数学：** 地图是训练域与测试域的比喻，图中的路线不代表测量曲线。本文起步实验是确定性函数外推；统计置信、模型校准与理论误差界需要各自的假设和证据，不能用同一张图互相替代。

> 状态：按64学时方案整理的可填写模板，正文、推导和各节完整实验待学生完成。已有漫画与起步代码是初始化材料，不代表学生成果。


### 第7章 引言：地图之外，还能相信吗？

想象一下，你雇了一位极其聪明的向导，他熟记了一片森林里每一条小径的路线。你让他在森林里带路，他几乎从未失手。这就像是人工智能在**训练数据所在的熟悉区域内**的表现——因为这是它“已经见过的区域”，它的每一个判断都有过往的经验作支撑，看起来无比可靠。

但此时此刻，你必须问自己一个严肃的问题：**当你们走出这片熟悉的森林，踏入一片向导从未见过的新区域时，你还能像之前那样信任他吗？**

在这个新区域里，向导可能依然气定神闲地为你指路，但他的指引可能通向绿洲，也可能通向悬崖。在科学计算和机器学习中，这正是我们面临的核心困境：**当AI（或数值模型）给出一个答案时，我们到底凭什么去相信它？**

仔细看这幅引入图，它其实描绘了我们建立“信任”时的三个关键锚点：

第一张图里，在 `[-1, 1]` 的**熟悉训练区域**内，模型的表现曲线是实线，这是有数据支撑的“舒适区”。但一旦我们来到第二张图，把目光投向 `[2, 3]` 这样**没有见过的新区域**时，曲线就变成了虚线和问号。此时，模型给出的答案不再是经验的延续，而是基于某种假设的“外推”。换了一个分布，模型内心其实是一无所知的。

这就引出了第三张图里那个直击灵魂的提问：“**地图外的路，还照旧走吗？**”

科学的态度绝不是盲从，也不是全盘否定，而是明白：**信任，是需要适用条件的。** 如果我们换了环境（分布变了），我们就必须重新寻找证据。

本章要做的核心工作，就是为你提供一套判断“信任边界”的科学工具。我们不讲空泛的哲学，而是要用严密的数学与实验，回答三个极其务实的问题：

1. **在条件理想的情况下，有没有一把标尺能保证模型在没见过的数据上依然可靠？**（我们要用泛化界来丈量风险的底线）
2. **如果环境真的变了，我们该如何设计独立的测试，来诚实地评估模型到底有没有“跨出地图”的能力？**（我们要厘清分布外泛化的真实边界）
3. **就算模型算得又快又好，我们怎么确认它真的解对了问题，且没有在自欺欺人？**（我们要用误差界与V&V体系，把代码计算和现实观测对上账）

记住，AI的答案之所以可信，从来不是因为它算得足够快、模型足够大，而是因为我们清楚地知道它**是在哪里学的、条件变没变，以及我们能用什么证据去检验它。**

现在，让我们带上这套寻找证据的工具，正式跨出熟悉的边界。



## 7.1 逼近理论与泛化界（2学时）

**本节目标：** 能区分表示能力、训练误差与对未见样本的保证。

本节的核心任务是在“表示能力”“训练误差”与“对未见样本的保证”三者之间建立严格的区分。为此，首先需要给出衡量这三者的数学对象，然后在一个条件足够清晰且可完全证明的设定下，建立经验风险与总体风险之间的高概率上界。

### 7.1.1 基本定义

**定义 7.1（假设类与逼近误差）** 设输入空间为 $\mathcal{X}$，输出空间为 $\mathcal{Y}$，目标函数为 $f:\mathcal{X}\to\mathcal{Y}$。假设类 $\mathcal{H}$ 是若干可测函数 $h:\mathcal{X}\to\mathcal{Y}$ 构成的集合。$\mathcal{H}$ 对 $f$ 的**逼近误差**定义为
$$
\inf_{h\in\mathcal{H}}\ \|h-f\|,
$$
其中 $\|\cdot\|$ 为依据具体问题选定的函数范数（如 $L^2$ 范数或一致范数）。

逼近误差仅由 $\mathcal{H}$ 与 $f$ 的几何关系决定，与样本数量及训练算法无关。它刻画的是假设类的**表示能力**：即使拥有无限数据与完美优化算法，若 $\mathcal{H}$ 中不存在接近 $f$ 的函数，学习过程亦无法产生低误差的预测器。

**定义 7.2（损失函数、经验风险与总体风险）** 设损失函数 $\ell:\mathcal{Y}\times\mathcal{Y}\to[0,1]$ 为可测且取值有界的函数。给定训练样本 $S=\{(x_1,y_1),\dots,(x_m,y_m)\}$，其中 $(x_i,y_i)$ 独立同分布于未知分布 $D$。对任意 $h\in\mathcal{H}$，**经验风险**定义为
$$
R_S(h)=\frac{1}{m}\sum_{i=1}^{m}\ell(h(x_i),y_i),
$$
**总体风险**定义为
$$
R(h)=\mathbb{E}_{(x,y)\sim D}\big[\ell(h(x),y)\big].
$$

经验风险是可由样本直接计算的量，总体风险是学习过程真正关心但无法直接计算的量。二者之间的差异构成了本节讨论的中心对象。

**定义 7.3（泛化界）** 给定置信参数 $\delta\in(0,1)$，若存在仅依赖于 $m$、$|\mathcal{H}|$ 与 $\delta$ 的函数 $\varepsilon(m,|\mathcal{H}|,\delta)$，使得对任意学习算法输出的假设 $h$，以至少 $1-\delta$ 的概率有
$$
R(h)\le R_S(h)+\varepsilon(m,|\mathcal{H}|,\delta),
$$
则称 $\varepsilon$ 为 $h$ 的一个**泛化界**。右端第二项通常可分解为“复杂度项”与“置信项”之和。

### 7.1.2 Hoeffding 不等式

以下引理是后续推导的基础。其证明可参见 Devroye、Györfi 与 Lugosi 的专著第 8.2 节。

**引理 7.1（Hoeffding 不等式）** 设 $Z_1,\dots,Z_m$ 为独立随机变量，且对每个 $i$，$Z_i\in[a,b]$ 几乎必然成立。令 $\bar{Z}=\frac{1}{m}\sum_{i=1}^{m}Z_i$。则对任意 $\varepsilon>0$，
$$
\mathbb{P}\big(|\bar{Z}-\mathbb{E}[\bar{Z}]|\ge\varepsilon\big)\le 2\exp\left(-\frac{2m\varepsilon^2}{(b-a)^2}\right).
$$
特别地，当 $Z_i\in[0,1]$ 时，
$$
\mathbb{P}\big(|\bar{Z}-\mathbb{E}[\bar{Z}]|\ge\varepsilon\big)\le 2\exp(-2m\varepsilon^2).
$$

### 7.1.3 单一假设的偏差界

先考虑退化情形：$\mathcal{H}$ 仅包含一个固定假设 $h$。此时样本不参与假设的选择，$h$ 在观测数据之前即已确定。

**定理 7.1（固定假设的泛化界）** 设损失 $\ell\in[0,1]$，$S$ 为从 $D$ 独立同分布抽取的 $m$ 个样本。对任意固定假设 $h$（不依赖于 $S$），以至少 $1-\delta$ 的概率有
$$
R(h)\le R_S(h)+\sqrt{\frac{\ln(2/\delta)}{2m}}.
$$

**证明** 对每个样本点定义 $Z_i=\ell(h(x_i),y_i)$。由于 $h$ 固定，$Z_1,\dots,Z_m$ 为独立同分布随机变量，取值于 $[0,1]$，且 $\mathbb{E}[Z_i]=R(h)$，$\frac{1}{m}\sum Z_i=R_S(h)$。由引理 7.1，
$$
\mathbb{P}\big(|R_S(h)-R(h)|\ge\varepsilon\big)\le 2e^{-2m\varepsilon^2}.
$$
令右端等于 $\delta$，解得 $\varepsilon=\sqrt{\ln(2/\delta)/(2m)}$。于是以概率至少 $1-\delta$ 有 $|R_S(h)-R(h)|<\varepsilon$，从而 $R(h)\le R_S(h)+\varepsilon$。$\square$

该定理的关键限制在于 $h$ 必须独立于训练样本。学习算法从数据中选择的假设恰恰违反这一条件：算法倾向于选择在 $S$ 上表现良好的假设，导致 $R_S(h)$ 系统性偏低。因此，定理 7.1 不能直接用于学习算法输出的假设。

### 7.1.4 有限假设类的一致收敛

为处理“假设由数据选出”带来的选择偏差，需要将偏差界对所有候选假设**同时**成立。有限假设类情形下，这一目标可通过联合界（union bound）实现。

**定理 7.2（有限假设类的一致收敛界）** 设 $\mathcal{H}$ 为有限假设类，$|\mathcal{H}|=M<\infty$，损失 $\ell\in[0,1]$，$S$ 为从 $D$ 独立同分布抽取的 $m$ 个样本。对任意 $\delta\in(0,1)$，以至少 $1-\delta$ 的概率，**对所有** $h\in\mathcal{H}$ 同时有
$$
R(h)\le R_S(h)+\sqrt{\frac{\ln M+\ln(1/\delta)}{2m}}.
$$
等价地，
$$
\mathbb{P}\left(\exists h\in\mathcal{H}:R(h)>R_S(h)+\sqrt{\frac{\ln M+\ln(1/\delta)}{2m}}\right)\le\delta.
$$

**证明** 固定任意 $h\in\mathcal{H}$。由定理 7.1，对 $\varepsilon>0$，
$$
\mathbb{P}\big(R(h)-R_S(h)\ge\varepsilon\big)\le e^{-2m\varepsilon^2}.
$$
（此处使用单侧偏差界，系数为 $1$ 而非 $2$。）对每个 $h$ 定义坏事件
$$
A_h=\left\{R(h)-R_S(h)\ge\varepsilon\right\}.
$$
由联合界，
$$
\mathbb{P}\left(\bigcup_{h\in\mathcal{H}}A_h\right)\le\sum_{h\in\mathcal{H}}\mathbb{P}(A_h)\le M e^{-2m\varepsilon^2}.
$$
令 $M e^{-2m\varepsilon^2}=\delta$，解得
$$
\varepsilon=\sqrt{\frac{\ln M+\ln(1/\delta)}{2m}}.
$$
在此 $\varepsilon$ 下，$\mathbb{P}(\bigcup_{h}A_h)\le\delta$，即至少以 $1-\delta$ 的概率，不存在 $h$ 使 $R(h)>R_S(h)+\varepsilon$。$\square$

该定理的假设条件需要逐项明确。第一，损失有界于 $[0,1]$；若损失有界于 $[0,B]$，则界中的平方根项需乘以 $B$。第二，样本独立同分布；若样本间存在相关性，Hoeffding 不等式不适用。第三，假设类有限；无限假设类需借助 VC 维或覆盖数等容量度量将 $\ln M$ 替换为相应复杂度项。第四，$\delta$ 为预先给定的置信参数，界以 $1-\delta$ 的概率成立，不构成逐样本保证。

**推论 7.1（经验风险最小化的泛化界）** 设 $\hat{h}_S=\arg\min_{h\in\mathcal{H}}R_S(h)$ 为经验风险最小化假设。在定理 7.2 的条件下，以至少 $1-\delta$ 的概率有
$$
R(\hat{h}_S)\le R_S(\hat{h}_S)+\sqrt{\frac{\ln M+\ln(1/\delta)}{2m}}.
$$

**证明** 定理 7.2 的结论对所有 $h\in\mathcal{H}$ 同时成立，$\hat{h}_S\in\mathcal{H}$，故结论对其成立。$\square$

推论 7.1 的意义在于：一旦一致收敛成立，经验风险最小化假设的总体风险可被其经验风险加上同一个与假设无关的惩罚项所上界。该惩罚项随 $M$ 增大而对数增长，随 $m$ 增大而以 $1/\sqrt{m}$ 速率衰减。

### 7.1.5 结构风险分解

将逼近误差引入后，可得到总体风险相对于假设类之外目标函数的完整分解。设 $h^*=\arg\min_{h\in\mathcal{H}}R(h)$ 为 $\mathcal{H}$ 中总体风险最小的假设。对任意学习算法输出的 $\hat{h}$，
$$
R(\hat{h})=\underbrace{R(h^*)}_{\text{逼近误差}}+\underbrace{\big(R(\hat{h})-R(h^*)\big)}_{\text{估计误差}}.
$$
逼近误差由 $\mathcal{H}$ 的表示能力决定，估计误差由有限样本下从 $\mathcal{H}$ 中选择 $\hat{h}$ 的统计代价决定。定理 7.2 所界定的正是估计误差中 $R(\hat{h})-R_S(\hat{h})$ 这一部分。若 $\hat{h}$ 在 $\mathcal{H}$ 中近似最小化总体风险，则估计误差主要由一致收敛项控制。然而，即使泛化界很紧，若逼近误差很大，总体风险仍可能很高。万能逼近定理所断言的是 $\mathcal{H}$ 足够丰富时逼近误差可任意小，但它不涉及估计误差，也不保证经验风险最小化在有限样本下能找到逼近误差意义下的最优函数。

### 7.1.6 适用边界

定理 7.2 的结论在以下条件不满足时不再成立。损失无界时，Hoeffding 不等式的指数衰减速率失效，需改用 sub-Gaussian 或 sub-exponential 集中不等式。样本非独立时，联合界与 Hoeffding 不等式均不能直接应用。假设类无限时，$\ln M$ 发散，界失去意义，需以 VC 维 $d_{\mathrm{VC}}$ 替换 $\ln M$，得到 $O(\sqrt{(d_{\mathrm{VC}}+\ln(1/\delta))/m})$ 形式的界。此外，该界对所有 $h\in\mathcal{H}$ 同时成立，因此对某个特定 $h$ 而言通常偏松，且不保证每个测试样本上的预测误差低于界值。

## 7.2 分布内与分布外泛化

7.1 节所建立的泛化界依赖于一个核心前提：训练样本与评价样本来自同一数据分布。一旦评价分布偏离训练分布，Hoeffding 不等式与一致收敛框架所依赖的独立同分布假设即告失效，7.1 节中的界在数学上不再适用。本节的任务是形式化这一偏离，并给出在分布改变条件下仍可使用的误差关系。

### 7.2.1 基本定义

**定义 7.4（域与风险）** 设输入空间 $\mathcal{X}$ 与输出空间 $\mathcal{Y}$。一个**域**由联合分布 $P(X,Y)$ 定义，其中 $X\in\mathcal{X}$，$Y\in\mathcal{Y}$。给定假设 $h:\mathcal{X}\to\mathcal{Y}$ 与损失函数 $\ell:\mathcal{Y}\times\mathcal{Y}\to\mathbb{R}_{\ge 0}$，假设 $h$ 在域 $P$ 上的**风险**定义为
$$
R_P(h)=\mathbb{E}_{(X,Y)\sim P}\big[\ell(h(X),Y)\big].
$$
设训练数据来自**源域** $P_S$，评价数据来自**目标域** $P_T$。当 $P_S=P_T$ 时，称评价为**分布内**；当 $P_S\ne P_T$ 时，称评价为**分布外**。

分布外的具体形式中，本节聚焦于**协变量偏移**：输入边际分布发生改变，但给定输入后的条件分布保持不变。这一假设在科学计算任务中具有明确对应：控制方程或物理规律不随参数取值改变，而参数的采样范围在训练与评价阶段不同。

**定义 7.5（协变量偏移）** 称源域 $P_S$ 与目标域 $P_T$ 之间发生**协变量偏移**，若
$$
P_S(Y\mid X)=P_T(Y\mid X)\quad \text{对几乎所有 } X,
$$
但边际分布 $P_S(X)\ne P_T(X)$。

在协变量偏移下，目标域风险与源域风险之间的关系完全由输入分布的差异与假设在输入空间上的行为决定。以下引入一个可估计的分布差异度量。

**定义 7.6（$\mathcal{H}$-散度）** 设 $\mathcal{H}$ 为取值于 $\{0,1\}$ 的假设类。定义对称差异假设类
$$
\mathcal{H}\Delta\mathcal{H}=\{h\oplus h':h,h'\in\mathcal{H}\},
$$
其中 $\oplus$ 表示异或运算。给定 $\mathcal{X}$ 上的两个分布 $P_X$ 与 $Q_X$，$\mathcal{H}\Delta\mathcal{H}$-**散度**定义为
$$
d_{\mathcal{H}\Delta\mathcal{H}}(P_X,Q_X)=2\sup_{h\in\mathcal{H}\Delta\mathcal{H}}\left|\mathbb{P}_{X\sim P_X}[h(X)=1]-\mathbb{P}_{X\sim Q_X}[h(X)=1]\right|.
$$

该量度量的是假设类 $\mathcal{H}$ 区分两个输入分布的能力。与总变差距离不同，$d_{\mathcal{H}\Delta\mathcal{H}}$ 受假设类复杂度的约束，因而可在有限样本下通过经验估计逼近。

### 7.2.2 协变量偏移下的目标风险界

以下定理给出了协变量偏移下目标域风险与源域风险之间的上界关系。其证明依赖于三角不等式与 $\mathcal{H}\Delta\mathcal{H}$-散度的定义。

**定理 7.3（Ben-David 等，2010）** 设 $\mathcal{H}$ 为取值于 $\{0,1\}$ 的假设类。对任意 $h\in\mathcal{H}$，
$$
R_{P_T}(h)\le R_{P_S}(h)+\frac{1}{2}d_{\mathcal{H}\Delta\mathcal{H}}(P_S(X),P_T(X))+\lambda,
$$
其中
$$
\lambda=\min_{h'\in\mathcal{H}}\big[R_{P_S}(h')+R_{P_T}(h')\big]
$$
称为**理想联合风险**。

**证明** 对任意 $h,h'\in\mathcal{H}$，由三角不等式，
$$
R_{P_T}(h)\le R_{P_T}(h')+R_{P_T}(h,h'),
$$
其中 $R_{P_T}(h,h')=\mathbb{E}_{P_T}[|h(X)-h'(X)|]$ 为 $h$ 与 $h'$ 在目标域上的差异。类似地，
$$
R_{P_S}(h,h')\le R_{P_S}(h)+R_{P_S}(h').
$$
将两式结合，
$$
R_{P_T}(h)\le R_{P_T}(h')+R_{P_S}(h,h')+R_{P_S}(h')-R_{P_S}(h,h')+R_{P_T}(h,h').
$$
对 $h'$ 取最小值，并注意到 $R_{P_T}(h,h')-R_{P_S}(h,h')$ 可被 $d_{\mathcal{H}\Delta\mathcal{H}}(P_S(X),P_T(X))/2$ 上界（因为 $h\oplus h'\in\mathcal{H}\Delta\mathcal{H}$，且 $|h-h'|=h\oplus h'$ 对二值函数成立），即得结论。$\square$

该定理的右端由三项构成：源域风险 $R_{P_S}(h)$、分布差异项 $\frac{1}{2}d_{\mathcal{H}\Delta\mathcal{H}}$、以及理想联合风险 $\lambda$。源域风险可在训练数据上经验估计；分布差异项可在无标签的源域与目标域输入样本上估计；$\lambda$ 反映的是是否存在一个假设能同时在两个域上表现良好。若不存在这样的假设，则无论分布差异多小，目标域风险都可能很大。

### 7.2.3 重要性加权

协变量偏移下的另一条技术路线是**重要性加权**。其基本思想是对源域样本赋予权重，使加权后的经验风险成为目标域风险的无偏估计。

**定义 7.7（重要性权重）** 在协变量偏移假设下，定义**重要性权重**为
$$
w(X)=\frac{P_T(X)}{P_S(X)}.
$$

**命题 7.1（重要性加权的无偏性）** 在协变量偏移下，对任意可积函数 $g$，
$$
\mathbb{E}_{P_S}[w(X)g(X,Y)]=\mathbb{E}_{P_T}[g(X,Y)].
$$

**证明** 由条件分布不变性 $P_S(Y\mid X)=P_T(Y\mid X)$，有
$$
\mathbb{E}_{P_S}[w(X)g(X,Y)]=\int w(x)g(x,y)P_S(x,y)\,dx\,dy
=\int \frac{P_T(x)}{P_S(x)}g(x,y)P_S(x)P_S(y\mid x)\,dx\,dy
$$
$$
=\int g(x,y)P_T(x)P_T(y\mid x)\,dx\,dy=\mathbb{E}_{P_T}[g(X,Y)].
$$
$\square$

因此，目标域风险可写为
$$
R_{P_T}(h)=\mathbb{E}_{P_S}[w(X)\ell(h(X),Y)].
$$
然而，重要性加权的实际困难在于 $w(X)$ 通常未知。直接估计两个输入密度再取比值在高维情形下方差极大[citation:3]。Sugiyama 等提出的直接重要性估计方法绕过了密度估计，通过最小化估计密度与真实密度之间的 Kullback-Leibler 散度来学习权重函数[citation:9]。此外，当 $P_S(X)$ 的支撑不能覆盖 $P_T(X)$ 的支撑时，$w(X)$ 在覆盖缺失区域无定义，重要性加权失效。

### 7.2.4 适用边界

定理 7.3 的界依赖于若干条件。第一，$\mathcal{H}$ 为二值假设类；对回归问题或一般有界损失，需将 $\mathcal{H}\Delta\mathcal{H}$-散度替换为相应的损失差异度量。第二，$\lambda$ 需要被独立估计或上界，而在实践中 $\lambda$ 本身不可直接计算，通常以源域与目标域上的经验联合风险近似。第三，该界不区分“分布外”的具体机制：若条件分布本身发生改变（概念漂移），协变量偏移假设不成立，定理 7.3 的前提即被破坏。

在科学计算语境中，协变量偏移是一种相对乐观的情形：控制方程的形式在训练与评价阶段保持一致，仅参数取值或初值条件发生变化。神经网络算子（如 DeepONet 与 FNO）在同分布样本上可达到极低的预测误差，但在高频或大幅值等分布外输入上误差显著增大[citation:17]。这一实验观察与理论界的含义一致：当目标域输入落在源域输入支撑的低密度区域或支撑之外时，$d_{\mathcal{H}\Delta\mathcal{H}}$ 或重要性权重趋于极端值，界失去实用意义。


## 7.3 误差界与 V&V

7.1 节与 7.2 节讨论的是统计意义下的泛化风险：给定分布与假设类，未见样本上的误差能以何种概率被界住。然而，一个更为基础的问题尚未被触及：**数值程序本身是否正确地求解了它所声称的方程？** 即便泛化界在统计上成立，若离散化实现中存在编程错误，或网格分辨率不足以控制截断误差，所计算的“经验风险”本身即为不可信的量。本节引入验证与确认（Verification and Validation, V&V）框架，形式化“解对方程”与“用对方程”之间的区分，并建立残差与解误差之间的严格关系。

### 7.3.1 基本定义

**定义 7.8（Verification 与 Validation）** 在计算科学与工程的 V&V 框架中，两个术语具有严格区分的含义[citation:12]。

**Verification（验证）** 是确定计算模型是否准确实现概念模型的数学描述的过程。它回答的问题是：程序是否正确地求解了所写的方程？Verification 进一步分为代码验证（code verification）与解验证（solution verification）。代码验证旨在发现编程错误，解验证旨在估计离散化引入的截断误差。

**Validation（确认）** 是确定计算模型在其适用范围内是否具有与预期用途相一致的足够精度的过程。它回答的问题是：所选的方程是否适合描述目标物理现象？Validation 必须依赖与真实物理观测的对比，而非仅依赖解析解或合成数据。

二者的关系可概括为：Verification 通过则说明“解对了方程”，Validation 通过则说明“选对了方程”。前者是后者的必要条件，但非充分条件。

**定义 7.9（制造解方法）** 制造解方法（Method of Manufactured Solutions, MMS）是代码验证的核心技术[citation:9][citation:14]。其构造如下：选取一个充分光滑的解析函数 $\bar{u}(x,t)$ 作为“制造解”，将其代入控制方程 $L(u)=0$，得到非零余项 $S=L(\bar{u})$。在原始方程中附加源项 $S$，使修正后的方程 $L(u)=S$ 以 $\bar{u}$ 为精确解。数值求解该修正方程，并将数值解与已知的 $\bar{u}$ 比较。若代码实现正确，离散误差应随网格加密按理论收敛阶下降。

制造解方法的意义在于：它将代码验证与物理真实性解耦。制造解无需具有物理意义，只需足够光滑以激发方程中的所有项。代码验证通过仅表明程序正确实现了离散格式，不表明该格式所求解的连续方程适合描述任何实际现象。

**定义 7.10（残差与解误差）** 考虑线性系统
$$
A x = b,
$$
其中 $A\in\mathbb{R}^{n\times n}$ 可逆，$b\in\mathbb{R}^n$。设 $\hat{x}$ 为数值方法给出的近似解。**残差**定义为
$$
r=b-A\hat{x},
$$
**解误差**定义为
$$
e=x-\hat{x}.
$$
残差可由计算直接得到，解误差在 $x$ 未知时不可直接计算。二者之间的关系由矩阵 $A$ 的性态决定。

### 7.3.2 残差与解误差的界

以下定理建立了残差范数与解误差范数之间的定量关系。其证明仅依赖矩阵范数的相容性与 $A$ 的可逆性。

**定理 7.4（残差-误差界）** 设 $A\in\mathbb{R}^{n\times n}$ 非奇异，$x$ 为 $Ax=b$ 的精确解，$\hat{x}$ 为近似解，$r=b-A\hat{x}$。则
$$
\frac{\|e\|}{\|x\|}\le\kappa(A)\frac{\|r\|}{\|b\|},
$$
其中 $\kappa(A)=\|A\|\|A^{-1}\|$ 为 $A$ 的条件数，$\|\cdot\|$ 为相容的向量范数与诱导矩阵范数。

**证明** 由 $r=b-A\hat{x}=Ax-A\hat{x}=A(x-\hat{x})=Ae$，得 $e=A^{-1}r$。取范数，
$$
\|e\|=\|A^{-1}r\|\le\|A^{-1}\|\|r\|.
$$
又由 $b=Ax$ 得 $\|b\|\le\|A\|\|x\|$，即 $\|x\|\ge\|b\|/\|A\|$。两式结合：
$$
\frac{\|e\|}{\|x\|}\le\frac{\|A^{-1}\|\|r\|}{\|b\|/\|A\|}=\|A\|\|A^{-1}\|\frac{\|r\|}{\|b\|}=\kappa(A)\frac{\|r\|}{\|b\|}.
$$
$\square$

该定理的结论具有明确的实践含义。条件数 $\kappa(A)$ 充当残差到解误差的**放大因子**。当 $\kappa(A)\approx 1$ 时，小残差蕴含小解误差。当 $\kappa(A)\gg 1$ 时，残差可能远小于解误差：一个残差极小的近似解仍可能与精确解相距甚远。因此，**残差小不构成解误差小的充分证据**，除非条件数受控。这与 7.1 节中经验风险小不保证总体风险小在结构上完全平行：两者都涉及“可计算量”与“目标量”之间被一个放大因子隔开的情形。

**命题 7.2（相对误差的双侧界）** 在定理 7.4 的条件下，进一步有
$$
\frac{1}{\kappa(A)}\frac{\|r\|}{\|b\|}\le\frac{\|e\|}{\|x\|}\le\kappa(A)\frac{\|r\|}{\|b\|}.
$$

**证明** 右半不等式即定理 7.4。左半不等式由 $\|r\|=\|Ae\|\le\|A\|\|e\|$ 与 $\|x\|=\|A^{-1}b\|\le\|A^{-1}\|\|b\|$ 得：
$$
\frac{\|r\|}{\|b\|}\le\frac{\|A\|\|e\|}{\|b\|}\le\frac{\|A\|\|e\|}{\|x\|/\|A^{-1}\|}=\kappa(A)\frac{\|e\|}{\|x\|}.
$$
移项即得。$\square$

命题 7.2 表明，条件数同时控制残差-误差关系的松紧程度。当 $\kappa(A)$ 很大时，双侧界之间的间隙可达 $\kappa(A)^2$ 倍，残差信息对解误差的指示能力极其有限。

### 7.3.3 网格收敛与观测阶

制造解方法的核心判据是数值格式的**观测收敛阶**应与理论收敛阶一致。设离散参数为 $h$（如网格间距），理论误差估计为
$$
\|u_h-u\| \le C h^p,
$$
其中 $p$ 为格式的理论阶数，$C$ 与 $h$ 无关。在制造解已知精确解 $\bar{u}$ 的情形下，对两个网格尺寸 $h_1>h_2$ 计算误差 $e_i=\|u_{h_i}-\bar{u}\|$，则观测阶可估计为
$$
p_{\text{obs}}=\frac{\ln(e_1/e_2)}{\ln(h_1/h_2)}.
$$
若 $p_{\text{obs}}$ 与理论 $p$ 在数值精度范围内一致，则代码实现通过验证[citation:9]。若 $p_{\text{obs}}$ 显著偏离 $p$，则可能存在编程错误、边界条件实现错误或离散格式不一致。网格加密序列的误差数据应呈对数线性下降趋势；偏离该趋势指示实现或求解过程中的异常。

### 7.3.4 适用边界

本节框架的适用性受以下条件约束。第一，定理 7.4 针对线性系统。非线性 PDE 的离散化通常需通过局部线性化或稳定性分析处理，残差与解误差的关系不再具有如此简洁的全局形式。第二，条件数的估计本身可能不可靠：对大规模稀疏矩阵，精确计算 $\kappa(A)$ 代价过高，通常只能获得下界估计。第三，制造解方法仅验证代码实现，不验证所求解的连续模型是否适合目标用途。代码验证通过后，Validation 仍需独立进行，其证据必须来自与可信物理观测的对比，而非合成数据。第四，本节讨论的误差界针对确定性离散误差；若训练数据本身含噪声或模型参数存在不确定性，还需结合 7.1 节的统计框架与 7.4 节的区间方法共同评估。


## 7.4 AI关联：不确定性量化

7.1 节至 7.3 节分别讨论了统计泛化界、分布偏移下的风险关系以及数值离散误差的界。这三类分析均以“点预测”为对象，即关注预测值偏离真实值的期望程度。然而，在科学决策场景中，仅给出点预测及其期望误差界尚不足以支撑风险判断。决策者需要知道：给定置信水平 $1-\alpha$，真实值落在哪个区间内？该区间是否在独立数据上达到标称覆盖率？本节引入不确定性量化的基本框架，并聚焦于一种具有有限样本覆盖保证的构造方法。

### 7.4.1 基本定义

**定义 7.11（随机不确定性与认知不确定性）** 设数据生成过程为 $(X,Y)\sim P$。在给定输入 $X=x$ 的条件下，输出 $Y$ 的**随机不确定性**（aleatoric uncertainty）指由条件分布 $P(Y\mid X=x)$ 自身的离散程度所决定的不可约波动。即使拥有关于 $P$ 的完全知识，该波动仍然存在。**认知不确定性**（epistemic uncertainty）指由于对 $P$ 的知识不完整或模型参数估计不精确所引入的不确定性，原则上可通过增加信息或数据予以减小。

需要指出，这一区分在操作层面存在边界模糊性。同一不确定性来源在不同建模视角下可能被归入不同类别，且二者的可分离性依赖于具体的概率模型设定。本节后续的区间构造不依赖于对这一区分的精确量化，而直接以覆盖率为目标。

**定义 7.12（预测区间与覆盖率）** 设 $(X_1,Y_1),\dots,(X_n,Y_n)$ 与 $(X_{n+1},Y_{n+1})$ 为独立同分布样本。一个**预测区间**是一个集值映射 $C:\mathcal{X}\to 2^{\mathcal{Y}}$，其中 $2^{\mathcal{Y}}$ 表示 $\mathcal{Y}$ 的幂集。给定显著性水平 $\alpha\in(0,1)$，若
$$
\mathbb{P}\big(Y_{n+1}\in C(X_{n+1})\big)\ge 1-\alpha,
$$
其中概率同时取于训练样本与测试点，则称 $C$ 具有**边际覆盖率**至少 $1-\alpha$。上述概率不条件于 $X_{n+1}=x$，因此边际覆盖不保证对每个 $x$ 的条件覆盖率。**区间宽度** $\mathbb{E}[\text{Leb}(C(X))]$ 度量区间的信息含量，与覆盖率共同构成评价预测区间质量的两个维度。

**定义 7.13（非一致性分数）** 设 $\mathcal{A}$ 为在训练集上拟合的预测模型。**非一致性分数**是一个可测函数 $s:\mathcal{X}\times\mathcal{Y}\to\mathbb{R}$，用于度量候选标签 $y$ 与输入 $x$ 在模型 $\mathcal{A}$ 下的“不一致程度”。常用的选择包括绝对残差 $s(x,y)=|y-\hat{\mu}(x)|$ 与归一化残差 $s(x,y)=|y-\hat{\mu}(x)|/\hat{\sigma}(x)$。

### 7.4.2 分裂共形预测的覆盖保证

分裂共形预测（split conformal prediction）将数据划分为训练集与校准集，在校准集上计算非一致性分数的经验分位数，并以此构造预测区间。其有限样本覆盖保证不依赖于数据分布的具体形式，仅要求样本可交换性。

**定义 7.14（分裂共形预测区间）** 设训练集 $\mathcal{D}_{\text{train}}$ 用于拟合模型 $\hat{\mu}$，校准集 $\mathcal{D}_{\text{cal}}=\{(X_i,Y_i)\}_{i=1}^{m}$ 与测试点 $(X_{n+1},Y_{n+1})$ 均未参与训练。对每个校准样本计算 $S_i=s(X_i,Y_i)$，将其按升序排列为 $S_{(1)}\le\cdots\le S_{(m)}$。给定显著性水平 $\alpha\in(0,1)$，定义
$$
k_\alpha=\lceil (m+1)(1-\alpha)\rceil.
$$
若 $k_\alpha\le m$，令 $\hat{q}_{1-\alpha}=S_{(k_\alpha)}$；若 $k_\alpha=m+1$，令 $\hat{q}_{1-\alpha}=+\infty$。分裂共形预测区间定义为
$$
C_{\alpha}(x)=\{y\in\mathcal{Y}:s(x,y)\le\hat{q}_{1-\alpha}\}.
$$

**定理 7.5（分裂共形的有限样本覆盖保证）** 设 $(X_1,Y_1),\dots,(X_m,Y_m),(X_{n+1},Y_{n+1})$ 为可交换随机变量。令 $C_\alpha$ 由定义 7.14 构造。则
$$
\mathbb{P}\big(Y_{n+1}\in C_\alpha(X_{n+1})\big)\ge 1-\alpha.
$$
此外，若非一致性分数几乎必然互不相同，则
$$
\mathbb{P}\big(Y_{n+1}\in C_\alpha(X_{n+1})\big)\le 1-\alpha+\frac{1}{m+1}.
$$

**证明** 将测试点的非一致性分数 $S_{n+1}=s(X_{n+1},Y_{n+1})$ 与校准分数 $S_1,\dots,S_m$ 合并。由可交换性，$S_{n+1}$ 在 $m+1$ 个分数中的秩均匀分布于 $\{1,\dots,m+1\}$。事件 $Y_{n+1}\notin C_\alpha(X_{n+1})$ 等价于 $S_{n+1}>\hat{q}_{1-\alpha}$。由 $\hat{q}_{1-\alpha}$ 的定义，$S_{n+1}>\hat{q}_{1-\alpha}$ 意味着 $S_{n+1}$ 的秩至少为 $k_\alpha+1$。秩大于等于 $k_\alpha+1$ 的概率为
$$
\frac{m+1-k_\alpha}{m+1}.
$$
由于 $k_\alpha=\lceil (m+1)(1-\alpha)\rceil\ge (m+1)(1-\alpha)$，有
$$
\frac{m+1-k_\alpha}{m+1}\le \frac{m+1-(m+1)(1-\alpha)}{m+1}=\alpha.
$$
因此 $\mathbb{P}(Y_{n+1}\notin C_\alpha(X_{n+1}))\le\alpha$，即覆盖概率至少 $1-\alpha$。上界由秩的离散性直接得出。$\square$

该定理的核心特征在于：覆盖保证对任意样本量 $m$ 成立，不依赖渐近近似；不对数据分布 $P$ 做参数假设；不要求模型 $\hat{\mu}$ 正确指定。然而，覆盖是边际的而非条件的：对某些 $X$ 取值，实际覆盖率可能低于 $1-\alpha$。此外，保证依赖于可交换性假设。当测试样本与校准样本的分布不一致时，可交换性被破坏，覆盖保证不再成立。这一限制与 7.2 节讨论的分布外问题直接对应：共形预测在分布外输入上的覆盖率没有理论保证。

**推论 7.2（区间宽度的单调性）** 在定理 7.5 的条件下，对固定的校准集，$\hat{q}_{1-\alpha}$ 随 $\alpha$ 减小而单调不减。因此，覆盖率要求越高（$\alpha$ 越小），区间宽度 $\mathbb{E}[\text{Leb}(C_\alpha(X))]$ 在期望意义上越大。

**证明** 若 $\alpha_1<\alpha_2$，则 $(m+1)(1-\alpha_1)>(m+1)(1-\alpha_2)$，故 $k_{\alpha_1}\ge k_{\alpha_2}$。由 $S_{(k)}$ 的单调性，$\hat{q}_{1-\alpha_1}\ge\hat{q}_{1-\alpha_2}$。由于 $C_\alpha(x)=\{y:s(x,y)\le\hat{q}_{1-\alpha}\}$，阈值越大则区间包含的 $y$ 越多，故宽度不减。$\square$

推论 7.2 的形式化表述揭示了覆盖率与宽度之间的根本权衡：任何声称高覆盖率的预测区间，必须同时报告其宽度，否则覆盖率信息不具有可比性。

### 7.4.3 适用边界

分裂共形预测的覆盖保证依赖于以下条件。第一，可交换性：校准样本与测试样本必须来自同一分布，且顺序不影响联合分布。在科学计算中，若测试任务的参数范围或初值条件超出校准集覆盖范围，可交换性不成立，定理 7.5 的结论失效。第二，非一致性分数的选择影响区间效率。以残差为分数时，区间宽度在异方差数据上可能过大；归一化分数可改善局部适应性，但需要可靠的尺度估计。第三，边际覆盖不蕴含条件覆盖。对特定的输入子群，覆盖率可能显著低于标称水平。Barber 等证明，在无额外假设的有限样本下，精确的条件覆盖不可能对任意分布实现。第四，共形预测仅校准点预测模型的不确定性表达，不改善模型本身的准确性；若基础模型在分布外区域系统性偏差，共形区间将以“覆盖真实值”为代价变得极宽或偏移。